importation des lib

In [136]:
import pandas as pd
import numpy as np

charger les donnnees

In [137]:
df = pd.read_csv('C:\\Users\\user\\Desktop\\financecore-transactions-analysis\\data\\brut\\bank-transactions.csv')

investigation des donnees

In [138]:
print(df.shape)
print(df.head())
print(df.dtypes)

(2060, 16)
  transaction_id client_id     date_transaction  montant devise  \
0      TXN000559   CLI0060  2022-04-19 02:31:00  2050.42    EUR   
1      TXN001154   CLI0057  2024-06-20 20:51:00  -123.66    GBP   
2      TXN000764   CLI0015  2024-08-28 05:03:00  -396.17    EUR   
3      TXN001598   CLI0045  2024-01-07 08:16:00    225.2    EUR   
4      TXN001873   CLI0034  2024-08-11 19:52:00   935.32    EUR   

   taux_change_eur  montant_eur      categorie              produit  \
0             1.00      2050.42  Depot especes       Compte Epargne   
1             0.86      -143.79    Retrait DAB  Credit Consommation   
2             1.00      -396.17    Prelevement                  PEA   
3             1.00       225.20    Paiement CB  Credit Consommation   
4             1.00       935.32       Interets    Credit Immobilier   

                 agence type_operation    statut  score_credit_client  \
0  Marseille-Vieux-Port         Credit  Complete                  NaN   
1            

donnees qualite

In [139]:
missing_ratio = df.isna().mean() * 100
print("\nMissing Values (%):\n", missing_ratio.sort_values(ascending=False))


Missing Values (%):
 taux_interet           100.000000
score_credit_client      8.106796
segment_client           5.097087
agence                   3.106796
transaction_id           0.000000
client_id                0.000000
date_transaction         0.000000
montant                  0.000000
categorie                0.000000
montant_eur              0.000000
taux_change_eur          0.000000
devise                   0.000000
statut                   0.000000
type_operation           0.000000
produit                  0.000000
solde_avant              0.000000
dtype: float64


In [140]:
duplicates = df.duplicated(subset='transaction_id').sum()
print("\nDuplicate transaction_id:", duplicates)


Duplicate transaction_id: 60


nettoyage des données

removation des doublons

In [141]:
df = df.sort_values(by='transaction_id')
df = df.drop_duplicates(subset='transaction_id', keep='first')

fixier la forme des date

In [142]:
df['date_transaction'] = pd.to_datetime(df['date_transaction'], errors='coerce' , dayfirst=True)

C:\Users\user\AppData\Local\Temp\ipykernel_14100\434822324.py:1: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['date_transaction'] = pd.to_datetime(df['date_transaction'], errors='coerce' , dayfirst=True)


nettoyage des montant

In [143]:
df['montant'] = df['montant'].astype(str).str.replace(',', '.', regex=False).astype(float)

nettoyage de solde_avant

In [144]:
df['solde_avant'] = df['solde_avant'].astype(str).str.replace(' EUR', '', regex=False).astype(float)

Normalize categorical columns

In [145]:
df['devise'] = df['devise'].str.upper().str.strip()
df['segment_client'] = df['segment_client'].str.capitalize().str.strip()
df['agence'] = df['agence'].str.strip()

gerer les valeurs manquants

In [146]:
df['score_credit_client'] = df['score_credit_client'].fillna(df['score_credit_client'].median())
df['segment_client'] = df['segment_client'].fillna(df['segment_client'].mode()[0])
df['agence'] = df['agence'].fillna(df['agence'].mode()[0])

detection des valeurs absurdes

In [147]:
Q1 = df['montant'].quantile(0.25)
Q3 = df['montant'].quantile(0.75)
IQR = Q3 - Q1
LOWER = Q1 - 1.5 * IQR
UPPER = Q3 + 1.5 * IQR
df['is.anomaly'] = (df['montant'] < LOWER) | (df['montant'] > UPPER)
df['is.anomaly'] = df['is.anomaly'] | (df['score_credit_client'] < 0) | (df['score_credit_client'] > 850)

INGÉNIERIE DES FONCTIONNALITÉS

In [148]:
df['annee'] = df['date_transaction'].dt.year
df['mois'] = df['date_transaction'].dt.month
df['trimestre'] = df['date_transaction'].dt.quarter
df['jour_semaine'] = df['date_transaction'].dt.day_name()

In [149]:
df['montant_eur_verifie'] = df['montant']  / df['taux_change_eur']

In [150]:
def risk_category(score):
    if score >= 700:
        return "Low"
    elif score >= 580:
        return "Medium"
    else:
        return "High"

df['categorie_risque'] = df['score_credit_client'].apply(risk_category)

In [160]:


# Supposons que votre DataFrame s'appelle df et contient les colonnes :
# 'client_id', 'montant', 'type_operation' ('credit' ou 'debit')

# Calcul du total des crédits et débits par client
solde_client = df.groupby('client_id').agg(
    total_credits = ('montant', lambda x: x[df.loc[x.index, 'type_operation'] == 'Credit'].sum()),
    total_debits  = ('montant', lambda x: x[df.loc[x.index, 'type_operation'] == 'Debit'].sum())
).reset_index()

# Calcul du solde net = total crédits - total débits
solde_client['solde_net'] = solde_client['total_credits'] - solde_client['total_debits']

# Affichage du résultat
solde_client.head()

,client_id,total_credits,total_debits,solde_net
0,CLI0001,12701.54,-11185.00,23886.54
1,CLI0002,3111.29,-7915.60,11026.89
2,CLI0003,5172.85,-15284.60,20457.45
3,CLI0004,7057.71,-8487.02,15544.73
4,CLI0005,1479.92,-9816.90,11296.82


In [151]:
client_agg = df.groupby('client_id').agg({
'transaction_id': 'count',
'montant': 'mean',
'produit': 'nunique'
}).reset_index()
client_agg.columns = ['client_id', 'nb_transactions', 'montant_moyen', 'nb_produits']
df = df.merge(client_agg, on='client_id', how='left')

In [159]:
print(df.isna().sum())
print(df.duplicated(subset='transaction_id').sum())
print(df.dtypes)

transaction_id         0
client_id              0
date_transaction       0
montant                0
devise                 0
taux_change_eur        0
montant_eur            0
categorie              0
produit                0
agence                 0
type_operation         0
statut                 0
score_credit_client    0
segment_client         0
solde_avant            0
is.anomaly             0
annee                  0
mois                   0
trimestre              0
jour_semaine           0
montant_eur_verifie    0
categorie_risque       0
nb_transactions        0
montant_moyen          0
nb_produits            0
dtype: int64
0
transaction_id                 object
client_id                      object
date_transaction       datetime64[ns]
montant                       float64
devise                         object
taux_change_eur               float64
montant_eur                   float64
categorie                      object
produit                        object
agence            

In [153]:
df = df.dropna(subset=['date_transaction'])


df['annee'] = df['date_transaction'].dt.year.astype(int)
df['mois'] = df['date_transaction'].dt.month.astype(int)
df['trimestre'] = df['date_transaction'].dt.quarter.astype(int)
df['jour_semaine'] = df['date_transaction'].dt.day_name()

In [154]:
df['annee'] = df['annee'].astype(int)
df['mois'] = df['mois'].astype(int)
df['trimestre'] = df['trimestre'].astype(int)

In [158]:
print(df.isna().sum())
print(df.duplicated(subset='transaction_id').sum())
print(df.dtypes)

transaction_id         0
client_id              0
date_transaction       0
montant                0
devise                 0
taux_change_eur        0
montant_eur            0
categorie              0
produit                0
agence                 0
type_operation         0
statut                 0
score_credit_client    0
segment_client         0
solde_avant            0
is.anomaly             0
annee                  0
mois                   0
trimestre              0
jour_semaine           0
montant_eur_verifie    0
categorie_risque       0
nb_transactions        0
montant_moyen          0
nb_produits            0
dtype: int64
0
transaction_id                 object
client_id                      object
date_transaction       datetime64[ns]
montant                       float64
devise                         object
taux_change_eur               float64
montant_eur                   float64
categorie                      object
produit                        object
agence            

In [156]:
df = df.drop(columns=['taux_interet'])

exportation des resultas

In [157]:
df.to_csv("../data/processed/financecore_clean.csv", index=False)